In [ ]:
import os
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import yaml
from models.afe_output_dynamic_range_characterization import (
    compute_adc_utilization,
    compute_gain_compression,
    evaluate_dc_operating_point,
    extract_waveform_metrics,
)
from PyLTSpice import RawRead  # type: ignore

%config InlineBackend.figure_format = 'svg'

In [ ]:
# ============================================
# Load Configuration
# ============================================
with open("config.yaml", "r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

# ============================================
# Output Directory Setup
# ============================================

REPORT_DIR = Path(config["paths"]["report_directory"])
REPORT_DIR.mkdir(exist_ok=True)

print(f"Reports directory ready at: {os.path.abspath(REPORT_DIR)}")

In [ ]:
"""
Clipping definition for ANA-003:

Clipping means:
- Output waveform flattening
- Output reaching supply limits
- Output no longer increasing proportionally with input

Clipping is NOT defined as:
- Output exceeding ADC full-scale peak

Exceeding ADC full-scale is expected when analog headroom
margin is intentionally designed. Clipping refers only to
loss of analog linear operating region.
"""

# ============================================================
# Configuration
# ============================================================

config: Dict = config

FULL_SCALE_VOLTAGE: float = float(config["adc"]["full_scale_voltage"])
FS_peak: float = FULL_SCALE_VOLTAGE / 2.0

VREF: float = float(config["v_ref"]["voltage"])
I_MAX_PHYSICAL: float = float(config["sensor"]["electrical"]["max_current_a"])

# UTIL_MIN and UTIL_MAX:
#   Allowed ADC utilization window at maximum specified detector current.
#   Ensures sufficient headroom while avoiding under-utilization.
#   These are hardware requirement limits.
UTIL_MIN: float = float(
    config["verification"]["ana_003"]["adc_utilization_min_percent"]
)
UTIL_MAX: float = float(
    config["verification"]["ana_003"]["adc_utilization_max_percent"]
)

# GAIN_COMP_MAX:
#   Maximum allowed gain compression at high-scale input.
#   Ensures the AFE remains within its linear operating region.
#   This is a hardware requirement limit.
GAIN_COMP_MAX: float = float(
    config["verification"]["ana_003"]["gain_compression_max_percent"]
)

# MIN_SWING_VPP:
#   Minimum required differential peak-to-peak swing of the AFE.
#   This ensures at least 20 percent headroom beyond ADC full-scale.
#   This is a hardware requirement derived from dynamic range budget.
MIN_SWING_VPP: float = float(
    config["verification"]["ana_003"]["differential_output_swing_min_vpp"]
)

# ============================================================
# Load Simulation Data
# ============================================================

raw_path: str = config["simulation"]["raw_file"]
raw: RawRead = RawRead(raw_path)

(
    input_amplitudes,
    output_peaks,
    output_baselines,
    output_common_mode,
) = extract_waveform_metrics(raw, config)

# ============================================================
# DC Operating Point Validation
# ============================================================

dc = evaluate_dc_operating_point(
    output_baselines,
    output_common_mode,
    FULL_SCALE_VOLTAGE,
    VREF,
)

if not dc["dc_check"]:
    raise RuntimeError("Invalid DC operating point.")

# ============================================================
# Dynamic Metrics Calculation
# ============================================================

# Global linear regression for nominal gain
coeffs = np.polyfit(input_amplitudes, output_peaks, 1)
gain_measured: float = float(coeffs[0])

# ADC utilization at maximum specified detector current
v_peak, utilization = compute_adc_utilization(
    gain_measured,
    I_MAX_PHYSICAL,
    FULL_SCALE_VOLTAGE,
)

# Gain compression using last sweep points
gain_full, gain_high, compression = compute_gain_compression(
    input_amplitudes,
    output_peaks,
)

# Maximum differential swing
max_peak: float = float(np.max(output_peaks))
max_vpp: float = 2.0 * max_peak

# ============================================================
# Acceptance Criteria Evaluation
# ============================================================

swing_ok: bool = max_vpp >= MIN_SWING_VPP
util_ok: bool = UTIL_MIN <= utilization <= UTIL_MAX
compression_ok: bool = compression <= GAIN_COMP_MAX

# Monotonicity check restricted to specified detector range
valid_region = input_amplitudes <= I_MAX_PHYSICAL
monotonic_ok: bool = bool(
    np.all(np.diff(output_peaks[valid_region]) > 0)
)
overall_pass: bool = swing_ok and util_ok and compression_ok and monotonic_ok

# ============================================================
# Reporting
# ============================================================

print("\n========== ANA-003 Output Dynamic Range ==========\n")

print(f"ADC full-scale peak: {FS_peak:.3f} V")
print(f"Output peak at max detector current: {v_peak:.3f} V")
print(f"ADC utilization: {utilization:.2f} percent")
print(f"Utilization window: {UTIL_MIN:.2f} to {UTIL_MAX:.2f} percent\n")

print(f"Maximum differential swing: {max_vpp:.3f} Vpp")
print(f"Minimum required swing: {MIN_SWING_VPP:.3f} Vpp\n")

print(f"Gain compression: {compression:.4f} percent")
print(f"Maximum allowed compression: {GAIN_COMP_MAX:.4f} percent\n")

print(f"Monotonic output within detector range: {monotonic_ok}")

print("\n========== FINAL RESULT ==========")
print(f"ANA-003: {'PASS' if overall_pass else 'FAIL'}")
print("==================================\n")

# ============================================================
# Plot 1 - ADC Utilization vs Input Current
# ============================================================

utilization_curve = output_peaks / FS_peak

plt.figure(figsize=(8, 6))
plt.plot(input_amplitudes * 1e3, utilization_curve, "o-", label="Measured")
plt.xlabel("Input Current [mA]")
plt.ylabel("Normalized Output (V_peak / V_FS_peak)")
plt.title("ANA-003 ADC Utilization")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(
        REPORT_DIR,
        "ANA003_OutputDynamicRange_Utilization.svg",
    )
)
plt.show()

# ============================================================
# Plot 2 - Gain Compression Behavior
# ============================================================

fit_full = np.polyval(coeffs, input_amplitudes)
residual = output_peaks - fit_full

plt.figure(figsize=(8, 6))
plt.plot(input_amplitudes * 1e3, residual * 1e3, "o-")
plt.xlabel("Input Current [mA]")
plt.ylabel("Deviation from Linear Fit [mV]")
plt.title("ANA-003 Gain Compression Behavior")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    os.path.join(
        REPORT_DIR,
        "ANA003_OutputDynamicRange_GainCompression.svg",
    )
)
plt.show()